In [ ]:
!git clone -q https://github.com/ynulihao/LLMRouterBench.git
%cd LLMRouterBench
!ls baselines baselines/adaptors
!cat requirements.txt

import pathlib, re
pat = {'gpu': r'cuda|\.to\(.*device|device_map', 'openai_api': r'openai|api_key|API_KEY',
       'pretrained_download': r'from_pretrained|hf_hub|SentenceTransformer', 'graph_libs': r'torch_geometric|dgl'}
for d in sorted(pathlib.Path('baselines').iterdir()):
    if d.is_dir():
        txt = ' '.join(p.read_text(errors='ignore') for p in d.rglob('*.py'))
        hits = {k: bool(re.search(v, txt, re.I)) for k, v in pat.items()}
        print(f'{d.name:20s}', hits)

In [ ]:
import pathlib
for d in ['AvengersPro', 'RouteLLM', 'FrugalGPT', 'GraphRouter', 'EmbedLLM', 'Best-route-llm']:
    p = pathlib.Path('baselines') / d
    print('=' * 20, d, '=' * 20)
    for f in ['requirements.txt']:
        if (p / f).exists(): print((p / f).read_text()[:600])
    for r in ['README.md', 'readme.md']:
        if (p / r).exists(): print((p / r).read_text()[:900])

print('=' * 20, 'config + data notes')
print(pathlib.Path('config/baseline_config_performance_cost.yaml').read_text()[:1500])
print(pathlib.Path('results/download.md').read_text()[:1500])
!grep -rniE "embedding|model *=" baselines/AvengersPro --include=*.py | head -15
!grep -niE "train|retrain|pretrain|checkpoint" baselines/adaptors/routellm_adaptor.py | head -15

In [ ]:
import pathlib
def show(path, n=1500):
    p = pathlib.Path(path); print('=' * 10, path)
    print(p.read_text(errors='ignore')[:n] if p.exists() else 'NOT FOUND')

!python --version
show('baselines/AvengersPro/config/simple_config_small_models_42.json')
!ls baselines/AvengersPro baselines/AvengersPro/config
!grep -rniE "EMBEDDING_API|base_url|EMBEDDING_BASE|embedding_model" baselines/AvengersPro baselines/adaptors --include=*.py | head -20
show('baselines/RouteLLM/mf_train_config.json')
!grep -rniE "OPENAI|embedding" baselines/RouteLLM --include=*.py | head -15
!ls baselines/FrugalGPT
show('baselines/FrugalGPT/run_train_local_scorer.sh')
!ls baselines/Best-route-llm
show('baselines/GraphRouter/README.md', 3500)

In [ ]:
import pathlib
def show(path, n=2500):
    p = pathlib.Path(path); print('=' * 10, path)
    print(p.read_text(errors='ignore')[:n] if p.exists() else 'NOT FOUND')

show('baselines/AvengersPro/config/simple_config.json')
show('baselines/AvengersPro/config/simple_config_small_models.json')
!grep -n "python" baselines/AvengersPro/README.md | head -30
!grep -n "add_argument" baselines/adaptors/avengerspro_adaptor.py
!ls results; du -sh results 2>/dev/null
!grep -noE "https?://[^) \"]*(huggingface|drive.google)[^) \"]*" README.md results/download.md
!nvidia-smi

In [ ]:
from huggingface_hub import HfApi
info = HfApi().dataset_info('NPULH/LLMRouterBench', files_metadata=True)
files = sorted([(s.size or 0, s.rfilename) for s in info.siblings], reverse=True)
print(f'{len(files)} files, total {sum(f[0] for f in files)/1e9:.2f} GB')
for sz, name in files[:15]: print(f'{sz/1e6:10.1f} MB  {name}')

!find . -name "embedding_config*.yaml"
!head -40 config/embedding_config.yaml
!sed -n '250,295p' baselines/adaptors/avengerspro_adaptor.py
!grep -nE "max_tokens|truncat|environ|EMBEDDING_API_KEY" baselines/AvengersPro/simple_cluster_router.py baselines/AvengersPro/config.py | head -25

In [ ]:
from huggingface_hub import HfApi
info = HfApi().dataset_info('withmartian/routerbench', files_metadata=True)
for s in sorted(info.siblings, key=lambda s: -(s.size or 0)):
    print(f'{(s.size or 0)/1e6:10.1f} MB  {s.rfilename}')

In [ ]:
from huggingface_hub import hf_hub_download
import pandas as pd, gc

p = hf_hub_download('withmartian/routerbench', 'routerbench_raw.pkl', repo_type='dataset')
raw = pd.read_pickle(p)           # only load pickles from sources you trust; this is the same repo you used before
print(raw.shape)
print('memory (GB):', round(raw.memory_usage(deep=True).sum() / 1e9, 2))

cols = list(raw.columns)
print('\nall columns:', cols[:60], '...' if len(cols) > 60 else '')
mix = [c for c in cols if 'mixtral' in c.lower()]
gpt = [c for c in cols if 'gpt-4-1106' in c.lower()]
print('\nMixtral columns:', mix)
print('GPT-4-1106 columns:', gpt)

for c in mix:
    s = raw[c]
    print(f'\n{c}: dtype={s.dtype}')
    if s.dtype == object:
        ex = s.dropna().iloc[0]
        print('  example type:', type(ex).__name__, '| length:', len(str(ex)))
        print('  example start:', repr(str(ex)[:300]))

In [ ]:
print(raw.model_name.value_counts().to_string())
print('\nduplicate (sample_id, model) pairs:', raw.duplicated(['sample_id', 'model_name']).sum())

mix = raw[raw.model_name.str.contains('mixtral', case=False)]
gpt = raw[raw.model_name.str.contains('gpt-4-1106', case=False)]
print('\nmixtral names:', mix.model_name.unique(), '| gpt-4-1106 names:', gpt.model_name.unique())

for label, d in [('mixtral', mix), ('gpt-4-1106', gpt)]:
    for ds in ['grade-school-math', 'arc-challenge']:
        sub = d[d.eval_name == ds]
        r = sub.model_response
        print(f'{label:11s} {ds:18s} rows={len(sub):5d}  null responses={r.isna().sum():4d}  '
              f'median length={r.dropna().astype(str).str.len().median()}')

ex = mix[mix.eval_name == 'grade-school-math'].iloc[0]
print('\nPROMPT  :', repr(ex.prompt[:300]))
print('RESPONSE:', repr(str(ex.model_response)[:400]))
print('performance:', ex.performance, '| cost:', ex.cost)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, numpy as np, pandas as pd, gc
folder = '/content/drive/MyDrive/MLResearch'
print(os.path.exists(f'{folder}/routerbench_scored_1000_all_routers.csv'))   # should print True

old = pd.read_csv(f'{folder}/routerbench_scored_1000_all_routers.csv')
if 'sample_id' in old.columns:
    chk = wide.add_prefix('raw_').rename(columns={'raw_sample_id': 'sample_id'})
    m_ = old.merge(chk, on='sample_id')
    print(f'matched {len(m_)} of {len(old)} prompts')
    print('strong score matches:', np.allclose(m_.score_strong, m_.raw_performance_strong),
          '| weak score matches:', np.allclose(m_.score_weak, m_.raw_performance_weak))
    print('strong cost matches :', np.allclose(m_.cost_strong, m_.raw_cost_strong),
          '| weak cost matches :', np.allclose(m_.cost_weak, m_.raw_cost_weak))
else:
    print('no sample_id column in the saved file:', list(old.columns))

wide.to_csv(f'{folder}/routerbench_gsm_arc_wide.csv', index=False)
del raw, sub, w; gc.collect()
print('saved and freed memory')

In [ ]:
import numpy as np, pandas as pd, gc

num = ['performance_strong', 'performance_weak', 'cost_strong', 'cost_weak']
wide[num] = wide[num].astype(float)
print(wide[num].dtypes.to_string())

chk = wide.add_prefix('raw_').rename(columns={'raw_sample_id': 'sample_id'})
m_ = old.merge(chk, on='sample_id')
print(f'\nmatched {len(m_)} of {len(old)} prompts')
print('strong score matches:', np.allclose(m_.score_strong, m_.raw_performance_strong),
      '| weak score matches:', np.allclose(m_.score_weak, m_.raw_performance_weak))
print('strong cost matches :', np.allclose(m_.cost_strong, m_.raw_cost_strong),
      '| weak cost matches :', np.allclose(m_.cost_weak, m_.raw_cost_weak))

wide.to_csv(f'{folder}/routerbench_gsm_arc_wide.csv', index=False)
del raw, sub, w; gc.collect()
print('saved and freed memory')

In [ ]:
import numpy as np, pandas as pd
pairs = [('score_strong', 'raw_performance_strong'), ('score_weak', 'raw_performance_weak'),
         ('cost_strong', 'raw_cost_strong'), ('cost_weak', 'raw_cost_weak')]
for a, b in pairs:
    x, y = m_[a].astype(float), m_[b].astype(float)
    d = (x - y).abs()
    print(f'{a:13s} vs raw: exactly equal {np.mean(x == y):.1%} | max diff {d.max():.6f} | '
          f'mean old {x.mean():.4f} / raw {y.mean():.4f} | corr {np.corrcoef(x, y)[0, 1]:.4f}')

print('\nby dataset (mean score, old vs raw):')
for ds_, g in m_.groupby('dataset'):
    print(f'  {ds_:18s} strong {g.score_strong.mean():.4f} / {g.raw_performance_strong.mean():.4f}'
          f' | weak {g.score_weak.mean():.4f} / {g.raw_performance_weak.mean():.4f}')

print('\ncost ratio raw/old (strong): median', (m_.raw_cost_strong / m_.cost_strong).median().round(4),
      '| weak:', (m_.raw_cost_weak / m_.cost_weak).median().round(4))

pc = [c for c in old.columns if 'prompt' in c.lower() or 'question' in c.lower()]
print('\nold columns with prompt/question text:', pc)
if 'prompt' in old.columns:
    same = (m_['prompt'].astype(str).str.strip() == m_['raw_prompt'].astype(str).str.strip())
    print('prompt text identical:', f'{same.mean():.1%}')
print('\nfirst 6 rows:')
print(m_[['sample_id', 'dataset', 'score_strong', 'raw_performance_strong',
          'score_weak', 'raw_performance_weak']].head(6).to_string(index=False))

In [ ]:
import numpy as np, pandas as pd
from huggingface_hub import hf_hub_download

# 1) where do the score differences sit?
m_['strong_diff'] = (m_.score_strong.astype(float) != m_.raw_performance_strong)
m_['weak_diff']   = (m_.score_weak.astype(float)   != m_.raw_performance_weak)
print(m_.groupby('dataset')[['strong_diff', 'weak_diff']].agg(['sum', 'mean']).round(3).to_string())

# 2) cost ratio, by dataset
for ds_, g in m_.groupby('dataset'):
    rs = float((g.raw_cost_strong / g.cost_strong).median()); rw = float((g.raw_cost_weak / g.cost_weak).median())
    cs = float(np.corrcoef(g.cost_weak, g.raw_cost_weak)[0, 1])
    print(f'{ds_:18s} median raw/old cost: strong {rs:.3f}, weak {rw:.3f} | weak-cost corr {cs:.3f}')

# 3) are the prompts the same text?
same = (m_['prompt'].astype(str).str.strip() == m_['raw_prompt'].astype(str).str.strip())
print('\nprompt text identical:', f'{same.mean():.1%}')
print(m_.loc[~same, ['sample_id', 'dataset']].head(5).to_string(index=False) if (~same).any() else '')

# 4) what does the dataset README say about the files?
rp = hf_hub_download('withmartian/routerbench', 'README.md', repo_type='dataset')
print('\n===== README =====')
print(open(rp).read()[:3500])

In [ ]:
import numpy as np, pandas as pd
from huggingface_hub import hf_hub_download
M, G = 'mistralai/mixtral-8x7b-chat', 'gpt-4-1106-preview'
z = pd.read_pickle(hf_hub_download('withmartian/routerbench', 'routerbench_0shot.pkl', repo_type='dataset'))
f = pd.read_pickle(hf_hub_download('withmartian/routerbench', 'routerbench_5shot.pkl', repo_type='dataset'))
print('0shot', z.shape, '| 5shot', f.shape)
print('0shot response-like columns:', [c for c in z.columns if 'response' in c.lower()][:6])
print('5shot eval_name counts (top):'); print(f.eval_name.value_counts().head(8).to_string())

ids = old.loc[old.dataset == 'arc-challenge', 'sample_id']
rw = wide.set_index('sample_id').loc[ids]
for name, t in [('0shot', z), ('5shot', f)]:
    s = t[t.sample_id.isin(ids)].set_index('sample_id')
    s = s.loc[s.index.intersection(rw.index)]
    r = rw.loc[s.index]
    print(f'\n{name}: {len(s)} of {len(ids)} ARC ids found')
    if len(s):
        print('  strong score == raw:', np.mean(s[G].astype(float).values == r.performance_strong.values).round(3),
              '| weak score == raw:', np.mean(s[M].astype(float).values == r.performance_weak.values).round(3))
        print('  median cost ratio raw/file (strong):', float((r.cost_strong / s[G + '|total_cost'].astype(float)).median()))
        print('  prompt text identical to raw:', np.mean(s.prompt.astype(str).str.strip().values == r.prompt.astype(str).str.strip().values).round(3))
        print('  mean prompt length (chars): file', s.prompt.astype(str).str.len().mean().round(0), '| raw', r.prompt.astype(str).str.len().mean().round(0))

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
M, G = 'mistralai/mixtral-8x7b-chat', 'gpt-4-1106-preview'
GSM, ARC = 'grade-school-math', 'arc-challenge'
SHARES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

def evaluate(s, w, cs, cw, b, shares):
    n_ = len(s); order = np.argsort(-b, kind='stable')
    a_s, a_w = s.mean(), w.mean(); out = []
    for sh in shares:
        k = int(round(sh * n_)); ts = np.zeros(n_, bool); ts[order[:k]] = True
        acc = np.where(ts, s, w).mean(); cost = np.where(ts, cs, cw).sum()
        racc = sh * a_s + (1 - sh) * a_w
        rcost = sh * cs.sum() + (1 - sh) * cw.sum()
        q = np.clip((acc - a_w) / (a_s - a_w), 0, 1) if a_s != a_w else 0.0
        crand = q * cs.sum() + (1 - q) * cw.sum()
        out.append((acc, cost, racc, rcost, crand - cost, crand, ts.mean()))
    return np.array(out)

# build the 5-shot version of the table: ARC rows from the 5-shot file, GSM8K unchanged
dup = f[f.sample_id.duplicated(keep=False)]
print('duplicate sample_ids in 5-shot file:', dup.sample_id.nunique(), '| differing scores among them:',
      int((dup.groupby('sample_id')[G].nunique() > 1).sum()))
f5 = f.drop_duplicates('sample_id').set_index('sample_id')
base = old.reset_index(drop=True)
d5 = base.copy()
arc = (d5.dataset == ARC).values
ids = d5.sample_id[arc]
assert set(ids) <= set(f5.index), 'some ARC ids missing from the 5-shot file'
d5.loc[arc, 'score_strong'] = f5.loc[ids, G].astype(float).values
d5.loc[arc, 'score_weak']   = f5.loc[ids, M].astype(float).values
d5.loc[arc, 'cost_strong']  = f5.loc[ids, G + '|total_cost'].astype(float).values
d5.loc[arc, 'cost_weak']    = f5.loc[ids, M + '|total_cost'].astype(float).values

R = {'BERT': base.bert_score.values, 'MF': base.mf_score.values, 'SW-ranking': base.sw_ranking_score.values}
rng = np.random.default_rng(42); B = 500
rows = []
for cond, d in [('ARC 0-shot (analysed so far)', base), ('ARC 5-shot (matches raw file)', d5)]:
    S, W, CS, CW, ds = (d.score_strong.values, d.score_weak.values, d.cost_strong.values,
                        d.cost_weak.values, d.dataset.values)
    print(f'{cond}: mean GPT-4 cost per prompt, GSM8K / ARC = {CS[ds == GSM].mean() / CS[ds == ARC].mean():.1f}x')
    for sub, mask in [('pooled', np.ones(len(d), bool)), ('ARC', ds == ARC), ('GSM8K', ds == GSM)]:
        ia = np.where(mask)[0]
        boots = [rng.choice(ia, len(ia)) for _ in range(B)]       # shared resamples for all routers
        y = (S[ia] > W[ia]).astype(int)
        for k, b in R.items():
            r = evaluate(S[ia], W[ia], CS[ia], CW[ia], b[ia], SHARES)
            gb, sb, ab = [], [], []
            for i in boots:
                rr = evaluate(S[i], W[i], CS[i], CW[i], b[i], SHARES)
                gb.append((rr[:, 0] - rr[:, 2]).mean()); sb.append(rr[:, 4].mean())
                yi = (S[i] > W[i]).astype(int)
                if len(set(yi)) == 2: ab.append(roc_auc_score(yi, b[i]))
            rows.append(dict(condition=cond, subset=sub, router=k,
                gain=(r[:, 0] - r[:, 2]).mean(), gain_lo=np.percentile(gb, 2.5), gain_hi=np.percentile(gb, 97.5),
                saved=r[:, 4].mean(), saved_lo=np.percentile(sb, 2.5), saved_hi=np.percentile(sb, 97.5),
                auc=roc_auc_score(y, b[ia]), auc_lo=np.percentile(ab, 2.5), auc_hi=np.percentile(ab, 97.5)))
out = pd.DataFrame(rows)
out.to_csv(f'{folder}/arc_0shot_vs_5shot_robustness.csv', index=False)
pd.set_option('display.width', 250)
for sub in ['ARC', 'pooled', 'GSM8K']:
    print(f'\n=== {sub} (mean over 9 cut-offs; 95% bootstrap intervals, 500 resamples)')
    t = out[out.subset == sub].copy()
    t['gain'] = [f'{a:+.4f} [{b:+.4f}, {c:+.4f}]' for a, b, c in zip(t.gain, t.gain_lo, t.gain_hi)]
    t['saved_usd'] = [f'{a:+.2f} [{b:+.2f}, {c:+.2f}]' for a, b, c in zip(t.saved, t.saved_lo, t.saved_hi)]
    t['auc'] = [f'{a:.3f} [{b:.3f}, {c:.3f}]' for a, b, c in zip(t.auc, t.auc_lo, t.auc_hi)]
    print(t[['condition', 'router', 'gain', 'saved_usd', 'auc']].to_string(index=False))

In [ ]:
cols = ['score_strong', 'score_weak', 'cost_strong', 'cost_weak']
print('NaN counts by dataset:')
print(d5.groupby('dataset')[cols].apply(lambda g: g.isna().sum()).to_string())
bad = d5[d5[cols].isna().any(axis=1)]
print(f'\n{len(bad)} rows with any NaN:', bad.dataset.value_counts().to_dict())
print(bad[['sample_id', 'dataset'] + cols].head(8).to_string(index=False))
chk_ids = bad.sample_id.head(5)
print('\nraw values in the 5-shot file for those ids:')
print(f5.loc[chk_ids, [G, M, G + '|total_cost', M + '|total_cost']].to_string())
print('\nscore dtype in 5-shot file:', f5[G].dtype, '| cost dtype:', f5[G + '|total_cost'].dtype)

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

for c in ['score_strong', 'score_weak']:
    d5[c] = pd.to_numeric(d5[c], errors='coerce')
bad_ids = d5.loc[d5[['score_strong', 'score_weak', 'cost_strong', 'cost_weak']].isna().any(axis=1), 'sample_id']
keep = ~d5.sample_id.isin(bad_ids).values
b0, b5 = base[keep].reset_index(drop=True), d5[keep].reset_index(drop=True)
print(f'dropped {int((~keep).sum())} prompts: {list(bad_ids)} | analysing {len(b0)}')

tie = np.random.default_rng(0).uniform(0, 1e-6, len(b0))
R = {'BERT': b0.bert_score.values, 'MF': b0.mf_score.values, 'SW-ranking': b0.sw_ranking_score.values,
     'Dataset-ID (GSM8K first)': (b0.dataset.values == GSM) + tie}

rng = np.random.default_rng(42); B = 500
rows = []
for cond, d in [('ARC 0-shot', b0), ('ARC 5-shot', b5)]:
    S, W, CS, CW, ds = (d.score_strong.values.astype(float), d.score_weak.values.astype(float),
                        d.cost_strong.values.astype(float), d.cost_weak.values.astype(float), d.dataset.values)
    assert not np.isnan(S).any() and not np.isnan(W).any() and not np.isnan(CS).any() and not np.isnan(CW).any()
    print(f'{cond}: GSM8K/ARC mean GPT-4 cost ratio = {CS[ds == GSM].mean() / CS[ds == ARC].mean():.1f}x'
          f' | ARC acc strong {S[ds == ARC].mean():.3f}, weak {W[ds == ARC].mean():.3f}')
    for sub, mask in [('pooled', np.ones(len(d), bool)), ('ARC', ds == ARC), ('GSM8K', ds == GSM)]:
        ia = np.where(mask)[0]
        boots = [rng.choice(ia, len(ia)) for _ in range(B)]
        y = (S[ia] > W[ia]).astype(int)
        for k, bsc in R.items():
            r = evaluate(S[ia], W[ia], CS[ia], CW[ia], bsc[ia], SHARES)
            gb, sb, ab = [], [], []
            for i in boots:
                rr = evaluate(S[i], W[i], CS[i], CW[i], bsc[i], SHARES)
                gb.append((rr[:, 0] - rr[:, 2]).mean()); sb.append(rr[:, 4].mean())
                yi = (S[i] > W[i]).astype(int)
                if len(set(yi)) == 2: ab.append(roc_auc_score(yi, bsc[i]))
            rows.append(dict(condition=cond, subset=sub, router=k,
                gain=(r[:, 0] - r[:, 2]).mean(), gain_lo=np.percentile(gb, 2.5), gain_hi=np.percentile(gb, 97.5),
                saved=r[:, 4].mean(), saved_lo=np.percentile(sb, 2.5), saved_hi=np.percentile(sb, 97.5),
                auc=roc_auc_score(y, bsc[ia]), auc_lo=np.percentile(ab, 2.5), auc_hi=np.percentile(ab, 97.5)))
out = pd.DataFrame(rows)
out.to_csv(f'{folder}/arc_0shot_vs_5shot_robustness.csv', index=False)
pd.set_option('display.width', 250)
for sub in ['ARC', 'pooled', 'GSM8K']:
    print(f'\n=== {sub} (mean over 9 cut-offs; 95% bootstrap intervals, {B} resamples)')
    t = out[out.subset == sub].copy()
    t['gain'] = [f'{a:+.4f} [{b:+.4f}, {c:+.4f}]' for a, b, c in zip(t.gain, t.gain_lo, t.gain_hi)]
    t['saved_usd'] = [f'{a:+.2f} [{b:+.2f}, {c:+.2f}]' for a, b, c in zip(t.saved, t.saved_lo, t.saved_hi)]
    t['auc'] = [f'{a:.3f} [{b:.3f}, {c:.3f}]' for a, b, c in zip(t.auc, t.auc_lo, t.auc_hi)]
    print(t[['condition', 'router', 'gain', 'saved_usd', 'auc']].to_string(index=False))

In [ ]:
import pandas as pd, numpy as np, gc
folder = '/content/drive/MyDrive/MLResearch'
M, G = 'mistralai/mixtral-8x7b-chat', 'gpt-4-1106-preview'
DS = ['grade-school-math', 'arc-challenge']

sub = raw[raw.model_name.isin([M, G]) & raw.eval_name.isin(DS)]
for (d, m), g in sub.groupby(['eval_name', 'model_name']):
    print(f'{d:18s} {m:28s} n={len(g)}  mean={g.performance.mean():.4f}  unique values: {sorted(g.performance.unique())[:8]}')

w = sub.pivot(index='sample_id', columns='model_name', values=['model_response', 'performance', 'cost'])
w.columns = [f'{a}_{"weak" if b == M else "strong"}' for a, b in w.columns]
meta = sub.drop_duplicates('sample_id').set_index('sample_id')[['eval_name', 'prompt']]
wide = meta.join(w).reset_index().rename(columns={'eval_name': 'dataset'})
print('\nwide table:', wide.shape, '| columns:', list(wide.columns))

# integrity check against the 1,000-prompt file you already scored
old = pd.read_csv(f'{folder}/routerbench_scored_1000_all_routers.csv')
if 'sample_id' in old.columns:
    chk = wide.add_prefix('raw_').rename(columns={'raw_sample_id': 'sample_id'})
    m_ = old.merge(chk, on='sample_id')
    print(f'\nmatched {len(m_)} of {len(old)} prompts')
    print('strong score matches:', np.allclose(m_.score_strong, m_.raw_performance_strong),
          '| weak score matches:', np.allclose(m_.score_weak, m_.raw_performance_weak))
    print('strong cost matches :', np.allclose(m_.cost_strong, m_.raw_cost_strong),
          '| weak cost matches :', np.allclose(m_.cost_weak, m_.raw_cost_weak))
else:
    print('\nno sample_id column in the saved file; paste old.columns here and I will adapt the check')

wide.to_csv(f'{folder}/routerbench_gsm_arc_wide.csv', index=False)
del raw, sub, w; gc.collect()
print('saved and freed memory')